# 06 - Advanced RAG Pipeline (Qdrant + Graph-aware retrieval)

Notebook advanced-only per pipeline RAG incrementale rispetto al naive:
- query rewriting/decomposition opzionale
- multi-retrieval + metadata filtering ibrido
- graph-aware expansion
- reranking deterministico
- benchmark MCQ + no-hint/judge con output metriche compatibili

## 1) Setup ambiente e import

Bootstrap del repository, import runtime advanced e utility benchmark condivise.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any
from dataclasses import asdict

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def _find_repo_root_for_bootstrap(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'notebooks' / 'pipelines' / 'common' / 'bootstrap.py').exists() and (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Project root non trovato. Avvia il notebook dentro il repository.')

_BOOTSTRAP_ROOT = _find_repo_root_for_bootstrap(Path.cwd().resolve())
if str(_BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(_BOOTSTRAP_ROOT))

from notebooks.pipelines.common.bootstrap import bootstrap_notebook
ROOT, SRC = bootstrap_notebook(start=Path.cwd().resolve())

from legal_indexing.rag_runtime import (
    AdvancedAnswerGuardConfig,
    AdvancedGraphExpansionConfig,
    AdvancedHybridConfig,
    AdvancedMetadataFilteringConfig,
    AdvancedMultiRetrievalConfig,
    AdvancedRagConfig,
    AdvancedRerankConfig,
    AdvancedRewriteConfig,
    McqAnswer,
    RagRuntimeConfig,
    align_record,
    build_naive_vs_advanced_comparison,
    build_rag_graph,
    load_valid_rows,
    level_sort_key,
    persist_benchmark_artifacts,
    persist_naive_vs_advanced_comparison,
    prepare_runtime,
    resolve_eval_positions,
    resolve_ollama_chat_url,
    run_advanced_benchmark,
    run_rag_question,
)

print('ROOT:', ROOT)
print('SRC:', SRC)


ROOT: /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite
SRC: /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/src


## 2) Configurazione runtime advanced e benchmark

Definiamo toggle mini/full benchmark e configurazione advanced modulare.

In [2]:
DATASET_DIR = ROOT / 'data' / 'laws_dataset_clean'
QDRANT_PATH = ROOT / 'data' / 'indexes' / 'qdrant'
INDEXING_ARTIFACTS_ROOT = ROOT / 'data' / 'qdrant_indexing'

EVAL_MCQ_CSV = ROOT / 'data' / 'evaluation' / 'questions.csv'
EVAL_NO_HINT_CSV = ROOT / 'data' / 'evaluation' / 'questions_no_hint.csv'

EXPLICIT_COLLECTION_NAME = None
INDEXING_RUN_ID = "20260302_211459"

RUN_FULL_BENCHMARK = True
BENCHMARK_START_POS = 0
BENCHMARK_LIMIT = 20
ENABLE_NAIVE_BASELINE = True
BENCHMARK_MAX_WORKERS = 4
BENCHMARK_CHECKPOINT_EVERY = 10
BENCHMARK_RESUME = True
BENCHMARK_POST_CHAT_MAX_RETRIES = 1
ADVANCED_PRESET = "quality_no_hint_v1"

QDRANT_URL = os.getenv('QDRANT_URL', 'http://127.0.0.1:6333')
QDRANT_PREFER_REMOTE = True

advanced_cfg = AdvancedRagConfig(
    hybrid=AdvancedHybridConfig(
        enabled=True,
        dense_top_k=12,
        sparse_top_k=20,
        fusion_method='rrf',
        rrf_k=60,
        dense_weight=1.0,
        sparse_weight=1.0,
        min_sparse_score=None,
        fallback_to_dense_only=True,
        query_analyzer='it_legal',
    ),
    rewrite=AdvancedRewriteConfig(
        enabled=True,
        use_llm=True,
        max_rewrites=1,
        max_subqueries=2,
        fallback_to_original=True,
    ),
    metadata_filtering=AdvancedMetadataFilteringConfig(
        mode='hybrid',
        enable_heuristics=True,
        relax_law_article_filters_on_relation_queries=True,
    ),
    multi_retrieval=AdvancedMultiRetrievalConfig(
        top_k_primary=6,
        top_k_secondary=3,
        dedupe_by_chunk_id=True,
    ),
    graph_expansion=AdvancedGraphExpansionConfig(
        enabled=True,
        max_related_laws=8,
        graph_retrieval_top_k=6,
        specific_query_mode='minimal',
        specific_query_max_related_laws=2,
        specific_query_graph_retrieval_top_k=2,
        include_related_articles=True,
        force_on_relation_queries=True,
    ),
    rerank=AdvancedRerankConfig(
        enabled=True,
        weight_retrieval_score=1.0,
        weight_graph_bonus=0.2,
        weight_metadata_bonus=0.25,
        weight_lexical_overlap=0.15,
        weight_sparse_score=0.2,
        tie_breaker='chunk_id',
    ),
    answer_guard=AdvancedAnswerGuardConfig(
        retry_on_empty_answer=True,
        max_empty_retries=1,
        retry_on_needs_more_context=False,
        max_needs_more_retries=0,
        mark_empty_as_pipeline_error=True,
    ),
    max_candidates=32,
)

config = RagRuntimeConfig(
    dataset_dir=DATASET_DIR,
    qdrant_path=QDRANT_PATH,
    qdrant_url=QDRANT_URL,
    qdrant_prefer_remote=QDRANT_PREFER_REMOTE,
    indexing_artifacts_root=INDEXING_ARTIFACTS_ROOT,
    indexing_run_id=INDEXING_RUN_ID,
    collection_name=EXPLICIT_COLLECTION_NAME,
    pipeline_mode='advanced',
    enforce_index_contract_coverage=True,
    index_contract_min_eval_coverage=0.95,
    view_filter='none',
    top_k=8,
    max_context_chunks=8,
    max_context_chars=9000,
    per_chunk_max_chars=900,
    llm_provider='pydanticai',
    advanced=advanced_cfg,
)
config.validate()

display({
    'pipeline_mode': config.pipeline_mode,
    'advanced_preset': ADVANCED_PRESET,
    'run_full_benchmark': RUN_FULL_BENCHMARK,
    'benchmark_start_pos': BENCHMARK_START_POS,
    'benchmark_limit': BENCHMARK_LIMIT,
    'enable_naive_baseline': ENABLE_NAIVE_BASELINE,
    'benchmark_max_workers': BENCHMARK_MAX_WORKERS,
    'benchmark_post_chat_max_retries': BENCHMARK_POST_CHAT_MAX_RETRIES,
    'benchmark_checkpoint_every': BENCHMARK_CHECKPOINT_EVERY,
    'benchmark_resume': BENCHMARK_RESUME,
    'hybrid_enabled': config.advanced.hybrid.enabled,
    'enforce_index_contract_coverage': config.enforce_index_contract_coverage,
    'index_contract_min_eval_coverage': config.index_contract_min_eval_coverage,
    'hybrid_fusion_method': config.advanced.hybrid.fusion_method,
    'qdrant_url': config.qdrant_url,
    'qdrant_prefer_remote': config.qdrant_prefer_remote,
    'dataset_dir': str(config.resolved_dataset_dir),
    'qdrant_path': str(config.resolved_qdrant_path),
    'indexing_artifacts_root': str(config.resolved_indexing_artifacts_root),
})

print('Nota operativa: evita di usare contemporaneamente Qdrant embedded e container Docker sullo stesso storage.')


{'pipeline_mode': 'advanced',
 'run_full_benchmark': True,
 'benchmark_start_pos': 0,
 'benchmark_limit': 20,
 'enable_naive_baseline': False,
 'benchmark_max_workers': 3,
 'benchmark_checkpoint_every': 10,
 'benchmark_resume': True,
 'hybrid_enabled': True,
 'enforce_index_contract_coverage': True,
 'index_contract_min_eval_coverage': 0.95,
 'hybrid_fusion_method': 'rrf',
 'qdrant_url': 'http://127.0.0.1:6333',
 'qdrant_prefer_remote': True,
 'dataset_dir': '/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/laws_dataset_clean',
 'qdrant_path': '/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/indexes/qdrant',
 'indexing_artifacts_root': '/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/qdrant_indexing'}

Nota operativa: evita di usare contemporaneamente Qdrant embedded e container Docker sullo stesso storage.


## 3) Preparazione runtime e validazioni

`prepare_runtime` valida dataset/contract/payload e apre risorse Qdrant.

In [3]:
runtime = prepare_runtime(config)

display({
    'dataset_valid': runtime.dataset_validation.is_valid,
    'dataset_counts': runtime.dataset_validation.counts,
    'resolved_collection': runtime.index_contract.collection_name,
    'resolved_run_id': runtime.index_contract.run_id,
    'collection_vector_size': runtime.collection_vector_size,
    'query_vector_size': runtime.query_vector_size,
    'dense_vector_name': runtime.dense_vector_name,
    'sparse_enabled': runtime.sparse_enabled,
    'sparse_vector_name': runtime.sparse_vector_name,
    'sparse_artifacts_path': runtime.sparse_artifacts_path,
})


{'dataset_valid': True,
 'dataset_counts': {'laws': 3145,
  'articles': 17774,
  'notes': 8345,
  'edges': 41400,
  'events': 26692,
  'chunks': 74517},
 'resolved_collection': 'laws_clean_da0144a2d28f_balanced_slurm_nomic-embed-te',
 'resolved_run_id': '20260302_211459',
 'collection_vector_size': 768,
 'query_vector_size': 768,
 'dense_vector_name': None,
 'sparse_enabled': True,
 'sparse_vector_name': 'bm25',
 'sparse_artifacts_path': '/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/qdrant_indexing/20260302_211459/sparse_encoder.json'}

## 4) Build grafo advanced e debug singola domanda

Compiliamo il grafo e ispezioniamo trace/diagnostica su una query di debug.

In [4]:
app = build_rag_graph(runtime.config, runtime)
print('LangGraph advanced compilato.')

DEBUG_QUESTION = 'Quali leggi hanno modificato la L.R. 11 febbraio 2020, n. 1?'
debug_run = run_rag_question(runtime.config, DEBUG_QUESTION, resources=runtime)
state = debug_run['state']

print('Trace nodes:', [x.get('node') for x in state.get('trace') or []])
print('Pipeline errors:', debug_run['pipeline_errors'])
display(pd.DataFrame(debug_run['retrieved_preview']))
display(debug_run['filters_summary'])
display(debug_run['graph_expansion'])
display(pd.DataFrame(debug_run['reranked']).head(10))
display(pd.DataFrame(debug_run['provenance_rows']).head(10))


LangGraph advanced compilato.
Trace nodes: ['normalize_query', 'rewrite_or_decompose_query', 'build_metadata_filter', 'retrieve_multi', 'graph_expand', 'rerank_candidates', 'build_context', 'generate_answer_structured']
Pipeline errors: []


,rank,chunk_id,law_id,article_id,law_status,law_date,score,source_passage_ids,source_chunk_ids,excerpt
0,1,vda:lr:2020-02-11:1#art:34#rc:1030000-1030000#...,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:34,current,2020-02-11,0.016129,[vda:lr:2020-02-11:1#art:34#p:c3],[vda:lr:2020-02-11:1#art:34#p:c3#chunk:0],"3. Le somme di cui al comma 2 sono previste, n..."
1,2,vda:lr:2020-02-11:1#art:39#rc:1060000-1060000#...,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:39,current,2020-02-11,0.016393,[vda:lr:2020-02-11:1#art:39#p:c6],[vda:lr:2020-02-11:1#art:39#p:c6#chunk:0],"6. Dopo l'articolo 18 della l.r. 19/2001 , com..."
2,3,vda:lr:2020-02-11:1#art:36#rc:0-0#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:36,current,2020-02-11,0.015873,[vda:lr:2020-02-11:1#art:36#p:intro],[vda:lr:2020-02-11:1#art:36#p:intro#chunk:0],(Disposizioni in materia di pesca. Modificazio...
3,4,vda:lr:2020-12-21:14#art:2#rc:1030000-1030000#...,vda:lr:2020-12-21:14,vda:lr:2020-12-21:14#art:2,current,2020-12-21,0.028191,[vda:lr:2020-12-21:14#art:2#p:c3],[vda:lr:2020-12-21:14#art:2#p:c3#chunk:0],"3. Dopo l'articolo 59 della l.r. 11/1998 , com..."
4,5,vda:lr:2020-02-11:1#art:4#rc:0-1010000#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:4,current,2020-02-11,0.016129,"[vda:lr:2020-02-11:1#art:4#p:intro, vda:lr:202...","[vda:lr:2020-02-11:1#art:4#p:intro#chunk:0, vd...",(Disposizioni in materia di assunzioni nel com...
5,6,vda:lr:2020-02-11:1#art:16#rc:0-1010000#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:16,current,2020-02-11,0.015873,"[vda:lr:2020-02-11:1#art:16#p:intro, vda:lr:20...","[vda:lr:2020-02-11:1#art:16#p:intro#chunk:0, v...",(Disposizioni in materia di organizzazione del...
6,7,vda:lr:2020-02-11:1#art:1#rc:0-0#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:1,current,2020-02-11,0.015625,[vda:lr:2020-02-11:1#art:1#p:intro],[vda:lr:2020-02-11:1#art:1#p:intro#chunk:0],(Interventi in materia di tasse regionali. Mod...
7,8,vda:lr:2020-12-21:14#art:10#rc:0-0#u:0#s:0,vda:lr:2020-12-21:14,vda:lr:2020-12-21:14#art:10,current,2020-12-21,0.016129,[vda:lr:2020-12-21:14#art:10#p:intro],[vda:lr:2020-12-21:14#art:10#p:intro#chunk:0],(Disposizioni in materia di regime dei beni de...
8,9,vda:lr:2020-02-11:1#art:12#rc:1050000-1050000#...,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:12,current,2020-02-11,0.016393,[vda:lr:2020-02-11:1#art:12#p:c5],[vda:lr:2020-02-11:1#art:12#p:c5#chunk:0],5. L'articolo 13 della legge regionale 8 april...
9,10,vda:lr:2020-02-11:1#art:45#rc:0-1010000#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:45,current,2020-02-11,0.015873,"[vda:lr:2020-02-11:1#art:45#p:intro, vda:lr:20...","[vda:lr:2020-02-11:1#art:45#p:intro#chunk:0, v...",(Dichiarazione d'urgenza) 1. La presente legge...


{'view_filter': 'none',
 'law_status_filter': None,
 'law_ids_filter': [],
 'metadata_seed_law_ids': ['vda:lr:2020-02-11:1'],
 'relation_types_filter': ['AMENDS', 'MODIFIED_BY'],
 'metadata_seed_article_ids': [],
 'year_from': 2020,
 'year_to': 2020,
 'metadata_mode': 'hybrid',
 'relation_query': True,
 'metadata_hard_law_filter_applied': False,
 'metadata_hard_article_filter_applied': False,
 'metadata_heuristics': ['relation_modific',
  'law_reference_resolved',
  'single_year_from_query'],
 'raw_filter_present': True}

{'seed_chunk_ids': ['vda:lr:2020-02-11:1#art:1#rc:0-0#u:0#s:0',
  'vda:lr:2020-02-11:1#art:12#rc:1050000-1050000#u:4#s:0',
  'vda:lr:2020-02-11:1#art:34#rc:1030000-1030000#u:2#s:0',
  'vda:lr:2020-02-11:1#art:36#rc:0-0#u:0#s:0',
  'vda:lr:2020-02-11:1#art:39#rc:1060000-1060000#u:6#s:0',
  'vda:lr:2020-02-11:1#art:45#rc:0-1010000#u:0#s:0',
  'vda:lr:2020-04-21:5#art:19#rc:1050000-1050000#u:5#s:0',
  'vda:lr:2020-04-21:5#art:7#rc:0-1010000#u:0#s:0',
  'vda:lr:2020-05-25:6#art:7#rc:0-1010000#u:0#s:0',
  'vda:lr:2020-12-21:14#art:10#rc:0-0#u:0#s:0',
  'vda:lr:2020-12-21:14#art:2#rc:1030000-1030000#u:6#s:0'],
 'seed_law_ids': ['vda:lr:2020-02-11:1',
  'vda:lr:2020-04-21:5',
  'vda:lr:2020-05-25:6',
  'vda:lr:2020-12-21:14'],
 'seed_article_ids': ['vda:lr:2020-02-11:1#art:1',
  'vda:lr:2020-02-11:1#art:12',
  'vda:lr:2020-02-11:1#art:34',
  'vda:lr:2020-02-11:1#art:36',
  'vda:lr:2020-02-11:1#art:39',
  'vda:lr:2020-02-11:1#art:45',
  'vda:lr:2020-04-21:5#art:19',
  'vda:lr:2020-04-21:5#art:

,chunk_id,retrieval_score,lexical_overlap,sparse_score,graph_bonus,metadata_bonus,final_score,source_tags
0,vda:lr:2020-02-11:1#art:34#rc:1030000-1030000#...,0.016129,0.333333,0.364249,1.0,1.000000,0.590525,"[graph, primary, primary_sparse, rewrite_1, re..."
1,vda:lr:2020-02-11:1#art:39#rc:1060000-1060000#...,0.016393,0.250000,0.375187,1.0,1.000000,0.585190,"[graph, primary, primary_sparse, rewrite_1, re..."
2,vda:lr:2020-02-11:1#art:36#rc:0-0#u:0#s:0,0.015873,0.250000,0.347982,1.0,1.000000,0.577868,"[graph, primary, primary_sparse, rewrite_1_spa..."
3,vda:lr:2020-12-21:14#art:2#rc:1030000-1030000#...,0.028191,0.333333,0.265040,1.0,0.666667,0.544451,"[graph, primary, primary_dense, primary_sparse..."
4,vda:lr:2020-02-11:1#art:4#rc:0-1010000#u:0#s:0,0.016129,0.333333,0.000000,1.0,1.000000,0.499462,"[graph, primary_dense, rewrite_2_dense]"
5,vda:lr:2020-02-11:1#art:16#rc:0-1010000#u:0#s:0,0.015873,0.166667,0.000000,1.0,1.000000,0.482540,[graph]
6,vda:lr:2020-02-11:1#art:1#rc:0-0#u:0#s:0,0.015625,0.166667,0.344370,0.0,1.000000,0.218384,"[primary, primary_sparse, rewrite_1_sparse, re..."
7,vda:lr:2020-12-21:14#art:10#rc:0-0#u:0#s:0,0.016129,0.250000,0.252401,0.0,0.666667,0.170896,"[primary_sparse, rewrite_2, rewrite_2_sparse]"
8,vda:lr:2020-02-11:1#art:12#rc:1050000-1050000#...,0.016393,0.166667,0.000000,0.0,1.000000,0.133060,"[rewrite_2, rewrite_2_dense]"
9,vda:lr:2020-02-11:1#art:45#rc:0-1010000#u:0#s:0,0.015873,0.166667,0.000000,0.0,1.000000,0.132540,"[primary, primary_dense, rewrite_1, rewrite_1_..."


,chunk_id,law_id,article_id,source_chunk_ids,source_passage_ids,score,cited
0,vda:lr:2020-02-11:1#art:34#rc:1030000-1030000#...,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:34,[vda:lr:2020-02-11:1#art:34#p:c3#chunk:0],[vda:lr:2020-02-11:1#art:34#p:c3],0.016129,False
1,vda:lr:2020-02-11:1#art:39#rc:1060000-1060000#...,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:39,[vda:lr:2020-02-11:1#art:39#p:c6#chunk:0],[vda:lr:2020-02-11:1#art:39#p:c6],0.016393,False
2,vda:lr:2020-02-11:1#art:36#rc:0-0#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:36,[vda:lr:2020-02-11:1#art:36#p:intro#chunk:0],[vda:lr:2020-02-11:1#art:36#p:intro],0.015873,False
3,vda:lr:2020-12-21:14#art:2#rc:1030000-1030000#...,vda:lr:2020-12-21:14,vda:lr:2020-12-21:14#art:2,[vda:lr:2020-12-21:14#art:2#p:c3#chunk:0],[vda:lr:2020-12-21:14#art:2#p:c3],0.028191,False
4,vda:lr:2020-02-11:1#art:4#rc:0-1010000#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:4,"[vda:lr:2020-02-11:1#art:4#p:intro#chunk:0, vd...","[vda:lr:2020-02-11:1#art:4#p:intro, vda:lr:202...",0.016129,False
5,vda:lr:2020-02-11:1#art:16#rc:0-1010000#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:16,"[vda:lr:2020-02-11:1#art:16#p:intro#chunk:0, v...","[vda:lr:2020-02-11:1#art:16#p:intro, vda:lr:20...",0.015873,False
6,vda:lr:2020-02-11:1#art:1#rc:0-0#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:1,[vda:lr:2020-02-11:1#art:1#p:intro#chunk:0],[vda:lr:2020-02-11:1#art:1#p:intro],0.015625,False
7,vda:lr:2020-12-21:14#art:10#rc:0-0#u:0#s:0,vda:lr:2020-12-21:14,vda:lr:2020-12-21:14#art:10,[vda:lr:2020-12-21:14#art:10#p:intro#chunk:0],[vda:lr:2020-12-21:14#art:10#p:intro],0.016129,False
8,vda:lr:2020-02-11:1#art:12#rc:1050000-1050000#...,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:12,[vda:lr:2020-02-11:1#art:12#p:c5#chunk:0],[vda:lr:2020-02-11:1#art:12#p:c5],0.016393,False
9,vda:lr:2020-02-11:1#art:45#rc:0-1010000#u:0#s:0,vda:lr:2020-02-11:1,vda:lr:2020-02-11:1#art:45,"[vda:lr:2020-02-11:1#art:45#p:intro#chunk:0, v...","[vda:lr:2020-02-11:1#art:45#p:intro, vda:lr:20...",0.015873,False


## 5) Caricamento benchmark condiviso e selezione posizioni

Allineiamo MCQ/no-hint e scegliamo subset mini (default) o full run via toggle.

In [5]:
mcq_rows = load_valid_rows(EVAL_MCQ_CSV)
no_hint_rows = load_valid_rows(EVAL_NO_HINT_CSV)

EVAL_POSITIONS = resolve_eval_positions(
    mcq_rows=mcq_rows,
    no_hint_rows=no_hint_rows,
    start_pos=BENCHMARK_START_POS,
    limit=BENCHMARK_LIMIT,
    run_full=RUN_FULL_BENCHMARK,
)

for pos in EVAL_POSITIONS:
    _ = align_record(pos, no_hint_rows, mcq_rows)

if not RUN_FULL_BENCHMARK:
    print('Nota: per baseline comparabile pre-tuning imposta RUN_FULL_BENCHMARK=True sul run_id corrente.')

display({
    'mcq_valid_rows': len(mcq_rows),
    'no_hint_valid_rows': len(no_hint_rows),
    'eval_start': EVAL_POSITIONS[0],
    'eval_end': EVAL_POSITIONS[-1],
    'eval_n': len(EVAL_POSITIONS),
    'mode': 'full' if RUN_FULL_BENCHMARK else 'mini',
})


{'mcq_valid_rows': 100,
 'no_hint_valid_rows': 100,
 'eval_start': 0,
 'eval_end': 99,
 'eval_n': 100,
 'mode': 'full'}

## 6) Setup chiamate structured per benchmark

Configuriamo endpoint chat OpenAI-compatible per MCQ e judge.

In [6]:
UTOPIA_API_KEY = os.getenv('UTOPIA_API_KEY', '')
if not UTOPIA_API_KEY:
    raise RuntimeError('UTOPIA_API_KEY mancante: necessario per benchmark structured')

BASE_URL = os.getenv('UTOPIA_BASE_URL', 'https://utopia.hpc4ai.unito.it')
API_URL = resolve_ollama_chat_url(BASE_URL)
CHAT_MODEL = os.getenv('UTOPIA_CHAT_MODEL', runtime.config.llm_model)
JUDGE_MODEL = os.getenv('UTOPIA_JUDGE_MODEL', CHAT_MODEL)
TIMEOUT_SEC = 120

HEADERS = {
    'Authorization': f'Bearer {UTOPIA_API_KEY}',
    'Content-Type': 'application/json',
}

print(json.dumps({
    'api_url': API_URL,
    'chat_model': CHAT_MODEL,
    'judge_model': JUDGE_MODEL,
    'timeout_sec': TIMEOUT_SEC,
}, ensure_ascii=False, indent=2))


{
  "api_url": "https://utopia.hpc4ai.unito.it/ollama/api/chat",
  "chat_model": "SLURM.gpt-oss:120b",
  "judge_model": "SLURM.gpt-oss:120b",
  "timeout_sec": 120
}


## 7) Esecuzione benchmark advanced (MCQ + no-hint/judge)

Usiamo il runner modulare in `src` con gestione errori per singola domanda.

In [7]:
checkpoint_dir = ROOT / 'notebooks' / 'rag_pipeline' / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
mode = 'full' if RUN_FULL_BENCHMARK else 'mini'
run_tag = f"{INDEXING_RUN_ID or 'auto'}_{mode}_start{BENCHMARK_START_POS}_n{len(EVAL_POSITIONS)}"
advanced_checkpoint_path = checkpoint_dir / f"benchmark_advanced_{run_tag}.jsonl"

benchmark_payload = run_advanced_benchmark(
    runtime=runtime,
    mcq_rows=mcq_rows,
    no_hint_rows=no_hint_rows,
    positions=EVAL_POSITIONS,
    api_url=API_URL,
    headers=HEADERS,
    chat_model=CHAT_MODEL,
    judge_model=JUDGE_MODEL,
    timeout_sec=TIMEOUT_SEC,
    max_workers=BENCHMARK_MAX_WORKERS,
    post_chat_max_retries=BENCHMARK_POST_CHAT_MAX_RETRIES,
    checkpoint_path=advanced_checkpoint_path,
    checkpoint_every=BENCHMARK_CHECKPOINT_EVERY,
    resume=BENCHMARK_RESUME,
)

mcq_summary = benchmark_payload['mcq_summary']
no_hint_summary = benchmark_payload['no_hint_summary']
comparison_table = benchmark_payload['comparison_table']
comparison_summary = benchmark_payload['comparison_summary']
recommended_workers = int((benchmark_payload.get('diagnostics') or {}).get('benchmark_recommended_max_workers_next_run') or BENCHMARK_MAX_WORKERS)
if recommended_workers != BENCHMARK_MAX_WORKERS:
    print(f"Nota throughput: prossimo run consigliato con BENCHMARK_MAX_WORKERS={recommended_workers} (da {BENCHMARK_MAX_WORKERS})")
    BENCHMARK_MAX_WORKERS = recommended_workers

display(pd.DataFrame(comparison_table['global_rows']))
display(pd.DataFrame(comparison_table['level_rows']))
print('Advanced checkpoint:', advanced_checkpoint_path)


,dataset,processed,judged,correct,accuracy,errors
0,MCQ,100,100,87,0.87,0
1,No-Hint + Judge,100,100,49,0.49,0


,level,mcq_correct,mcq_judged,mcq_accuracy,no_hint_correct,no_hint_judged,no_hint_accuracy,delta_no_hint_minus_mcq
0,L1,24,25,0.96,15,25,0.60,-0.36
1,L2,20,25,0.80,13,25,0.52,-0.28
2,L3,20,25,0.80,9,25,0.36,-0.44
3,L4,23,25,0.92,12,25,0.48,-0.44


Advanced checkpoint: /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/notebooks/rag_pipeline/checkpoints/benchmark_advanced_20260302_211459_full_start0_n100.jsonl


## 8) Benchmark naive di controllo per confronto


In [8]:
naive_runtime = None
naive_benchmark_payload = None
naive_comparison = None
naive_checkpoint_path = None

if ENABLE_NAIVE_BASELINE:
    naive_config = config.with_overrides(pipeline_mode='naive')
    naive_runtime = prepare_runtime(naive_config, client=runtime.client)
    naive_checkpoint_path = checkpoint_dir / f"benchmark_naive_{run_tag}.jsonl"

    naive_benchmark_payload = run_advanced_benchmark(
        runtime=naive_runtime,
        mcq_rows=mcq_rows,
        no_hint_rows=no_hint_rows,
        positions=EVAL_POSITIONS,
        api_url=API_URL,
        headers=HEADERS,
        chat_model=CHAT_MODEL,
        judge_model=JUDGE_MODEL,
        timeout_sec=TIMEOUT_SEC,
        max_workers=BENCHMARK_MAX_WORKERS,
        post_chat_max_retries=BENCHMARK_POST_CHAT_MAX_RETRIES,
        checkpoint_path=naive_checkpoint_path,
        checkpoint_every=BENCHMARK_CHECKPOINT_EVERY,
        resume=BENCHMARK_RESUME,
    )

    naive_comparison = build_naive_vs_advanced_comparison(
        naive_payload=naive_benchmark_payload,
        advanced_payload=benchmark_payload,
    )

    display(pd.DataFrame([naive_comparison['global']]))
    display(pd.DataFrame(naive_comparison['by_level']))
    print('Naive checkpoint:', naive_checkpoint_path)
else:
    print('Naive baseline disabilitato: verra eseguito solo nel confronto finale.')


Naive baseline disabilitato: verra eseguito solo nel confronto finale.


## 9) Output metriche compatibili (shape 02/05)


In [9]:
summary_payload = {
    'advanced': {
        'mcq': {
            'processed': mcq_summary['processed'],
            'judged': mcq_summary['judged'],
            'accuracy': mcq_summary['accuracy'],
            'errors': mcq_summary['errors'],
        },
        'no_hint': {
            'processed': no_hint_summary['processed'],
            'judged': no_hint_summary['judged'],
            'accuracy': no_hint_summary['accuracy'],
            'errors': no_hint_summary['errors'],
        },
    },
    'advanced_diagnostics': benchmark_payload.get('diagnostics') or {},
}

if naive_benchmark_payload is not None and naive_comparison is not None:
    summary_payload['naive'] = {
        'mcq_accuracy': (naive_benchmark_payload.get('mcq_summary') or {}).get('accuracy'),
        'no_hint_accuracy': (naive_benchmark_payload.get('no_hint_summary') or {}).get('accuracy'),
    }
    summary_payload['advanced_vs_naive'] = naive_comparison['global']

print('=== GLOBAL SUMMARY ===')
print(json.dumps(summary_payload, ensure_ascii=False, indent=2))


=== GLOBAL SUMMARY ===
{
  "advanced": {
    "mcq": {
      "processed": 100,
      "judged": 100,
      "accuracy": 0.87,
      "errors": 0
    },
    "no_hint": {
      "processed": 100,
      "judged": 100,
      "accuracy": 0.49,
      "errors": 0
    }
  },
  "advanced_diagnostics": {
    "benchmark_max_workers": 3,
    "benchmark_post_chat_max_retries": 2,
    "benchmark_checkpoint_path": "/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/notebooks/rag_pipeline/checkpoints/benchmark_advanced_20260302_211459_full_start0_n100.jsonl",
    "benchmark_checkpoint_resume": true,
    "mcq_checkpoint_loaded_rows": 0,
    "no_hint_checkpoint_loaded_rows": 0,
    "mcq_pipeline_error_rows": 0,
    "no_hint_pipeline_error_rows": 20,
    "no_hint_empty_detected_count": 19,
    "no_hint_fallback_used_count": 0,
    "no_hint_empty_by_level": {
      "L1": {
        "processed": 25,
        "empty_detected": 4,
        "fallback_used": 0,
 

## 10) Persistenza artifact benchmark


In [10]:
artifacts_dir = ROOT / 'notebooks' / 'rag_pipeline' / 'artifacts'
mode = 'full' if RUN_FULL_BENCHMARK else 'mini'

advanced_artifacts = persist_benchmark_artifacts(
    artifacts_dir=artifacts_dir,
    label='advanced',
    mode=mode,
    config_payload={
        'pipeline_mode': runtime.config.pipeline_mode,
        'advanced_preset': ADVANCED_PRESET,
        'view_filter': runtime.config.view_filter,
        'top_k': runtime.config.top_k,
        'max_context_chunks': runtime.config.max_context_chunks,
        'max_context_chars': runtime.config.max_context_chars,
        'per_chunk_max_chars': runtime.config.per_chunk_max_chars,
        'benchmark_start_pos': BENCHMARK_START_POS,
        'benchmark_limit': BENCHMARK_LIMIT,
        'benchmark_n': len(EVAL_POSITIONS),
        'run_full_benchmark': RUN_FULL_BENCHMARK,
        'benchmark_max_workers': BENCHMARK_MAX_WORKERS,
        'benchmark_post_chat_max_retries': BENCHMARK_POST_CHAT_MAX_RETRIES,
        'benchmark_checkpoint_every': BENCHMARK_CHECKPOINT_EVERY,
        'benchmark_resume': BENCHMARK_RESUME,
        'qdrant_url': runtime.config.qdrant_url,
        'qdrant_prefer_remote': runtime.config.qdrant_prefer_remote,
        'enforce_index_contract_coverage': runtime.config.enforce_index_contract_coverage,
        'index_contract_min_eval_coverage': runtime.config.index_contract_min_eval_coverage,
        'advanced': asdict(runtime.config.advanced),
    },
    index_contract=runtime.index_contract.to_dict(),
    benchmark_payload=benchmark_payload,
)

naive_artifacts = None
comparison_artifact = None
if naive_benchmark_payload is not None and naive_runtime is not None and naive_comparison is not None:
    naive_artifacts = persist_benchmark_artifacts(
        artifacts_dir=artifacts_dir,
        label='naive',
        mode=mode,
        config_payload={
            'pipeline_mode': naive_runtime.config.pipeline_mode,
            'advanced_preset': ADVANCED_PRESET,
            'view_filter': naive_runtime.config.view_filter,
            'top_k': naive_runtime.config.top_k,
            'max_context_chunks': naive_runtime.config.max_context_chunks,
            'max_context_chars': naive_runtime.config.max_context_chars,
            'per_chunk_max_chars': naive_runtime.config.per_chunk_max_chars,
            'benchmark_start_pos': BENCHMARK_START_POS,
            'benchmark_limit': BENCHMARK_LIMIT,
            'benchmark_n': len(EVAL_POSITIONS),
            'run_full_benchmark': RUN_FULL_BENCHMARK,
            'benchmark_max_workers': BENCHMARK_MAX_WORKERS,
            'benchmark_post_chat_max_retries': BENCHMARK_POST_CHAT_MAX_RETRIES,
            'benchmark_checkpoint_every': BENCHMARK_CHECKPOINT_EVERY,
            'benchmark_resume': BENCHMARK_RESUME,
            'qdrant_url': naive_runtime.config.qdrant_url,
            'qdrant_prefer_remote': naive_runtime.config.qdrant_prefer_remote,
            'enforce_index_contract_coverage': naive_runtime.config.enforce_index_contract_coverage,
            'index_contract_min_eval_coverage': naive_runtime.config.index_contract_min_eval_coverage,
            'advanced': asdict(runtime.config.advanced),
        },
        index_contract=naive_runtime.index_contract.to_dict(),
        benchmark_payload=naive_benchmark_payload,
    )

    comparison_artifact = persist_naive_vs_advanced_comparison(
        artifacts_dir=artifacts_dir,
        comparison_payload=naive_comparison,
    )

print('Advanced JSON:', advanced_artifacts['json'])
if naive_artifacts is not None:
    print('Naive JSON:', naive_artifacts['json'])
if comparison_artifact is not None:
    print('Comparison JSON:', comparison_artifact)


Advanced JSON: /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/notebooks/rag_pipeline/artifacts/rag_advanced_full_benchmark_20260303_120551.json


## 11) Dashboard e analisi errori e confronto Naive vs Advanced

In questa sezione visualizziamo: accuracy globale/per livello, failure taxonomy, diagnostica hybrid, scatter diagnostici e coverage del contract indice (eval references covered vs missing).


In [1]:
if naive_benchmark_payload is None:
    print('Grafici comparativi Naive vs Advanced saltati (ENABLE_NAIVE_BASELINE=False).')
else:
    mcq_results = benchmark_payload['mcq_results']
    no_hint_results = benchmark_payload['no_hint_results']

    adv_mcq_summary = benchmark_payload['mcq_summary']
    adv_no_hint_summary = benchmark_payload['no_hint_summary']
    naive_mcq_summary = naive_benchmark_payload['mcq_summary']
    naive_no_hint_summary = naive_benchmark_payload['no_hint_summary']

    all_levels = sorted(
        set(adv_no_hint_summary['by_level'].keys()) | set(naive_no_hint_summary['by_level'].keys()),
        key=level_sort_key,
    )

    # 1) Accuracy globale Naive vs Advanced (MCQ e No-Hint)
    global_labels = ['MCQ', 'No-Hint + Judge']
    naive_global = [
        naive_mcq_summary.get('accuracy') or 0.0,
        naive_no_hint_summary.get('accuracy') or 0.0,
    ]
    adv_global = [
        adv_mcq_summary.get('accuracy') or 0.0,
        adv_no_hint_summary.get('accuracy') or 0.0,
    ]

    # 2) Accuracy per livello No-Hint
    naive_level_no_hint = [
        (naive_no_hint_summary['by_level'].get(lvl, {}).get('accuracy') or 0.0)
        for lvl in all_levels
    ]
    adv_level_no_hint = [
        (adv_no_hint_summary['by_level'].get(lvl, {}).get('accuracy') or 0.0)
        for lvl in all_levels
    ]

    adv_no_hint_df = pd.DataFrame(no_hint_results)
    if adv_no_hint_df.empty:
        adv_no_hint_df = pd.DataFrame(
            columns=[
                'level', 'failure_category', 'retrieval_mode',
                'dense_retrieved_count', 'sparse_retrieved_count',
                'fusion_overlap_count', 'reference_law_hit',
                'context_included_count', 'final_binary_score',
                'rag_was_empty_before_guard', 'rag_answer_source',
                'top1_law_match', 'context_precision_proxy',
            ]
        )

    for col, default in {
        'level': '',
        'failure_category': '',
        'retrieval_mode': 'dense_only',
        'dense_retrieved_count': 0,
        'sparse_retrieved_count': 0,
        'fusion_overlap_count': 0,
        'context_included_count': 0,
        'final_binary_score': 0,
        'rag_was_empty_before_guard': False,
        'rag_answer_source': '',
        'top1_law_match': None,
        'context_precision_proxy': None,
    }.items():
        if col not in adv_no_hint_df.columns:
            adv_no_hint_df[col] = default

    adv_no_hint_df['level'] = adv_no_hint_df['level'].astype(str)
    adv_no_hint_df['sparse_contribution'] = (
        adv_no_hint_df['sparse_retrieved_count'].astype(float)
        / (
            adv_no_hint_df['dense_retrieved_count'].astype(float)
            + adv_no_hint_df['sparse_retrieved_count'].astype(float)
            + 1e-9
        )
    )

    # 3) Failure decomposition stacked bar (Advanced, no-hint)
    failure_categories = ['retrieval_miss', 'context_noise', 'abstention', 'contradiction']
    failure_by_level = {lvl: {cat: 0 for cat in failure_categories} for lvl in all_levels}
    for _, row in adv_no_hint_df.iterrows():
        lvl = str(row.get('level') or '')
        cat = str(row.get('failure_category') or '')
        if lvl in failure_by_level and cat in failure_categories:
            failure_by_level[lvl][cat] += 1

    # 4) Hybrid diagnostics per livello
    hybrid_by_level = []
    for lvl in all_levels:
        subset = adv_no_hint_df[adv_no_hint_df['level'] == lvl]
        if subset.empty:
            hybrid_by_level.append({'level': lvl, 'dense': 0.0, 'sparse': 0.0, 'overlap': 0.0, 'ref_hit_rate': 0.0})
            continue
        ref_known = subset['reference_law_hit'].dropna()
        ref_hit_rate = float(ref_known.astype(bool).mean()) if not ref_known.empty else 0.0
        hybrid_by_level.append({
            'level': lvl,
            'dense': float(subset['dense_retrieved_count'].mean()),
            'sparse': float(subset['sparse_retrieved_count'].mean()),
            'overlap': float(subset['fusion_overlap_count'].mean()),
            'ref_hit_rate': ref_hit_rate,
        })
    hybrid_df = pd.DataFrame(hybrid_by_level)

    # 5) Empty/fallback diagnostics
    empty_rate_by_level = []
    fallback_count_by_level = []
    for lvl in all_levels:
        subset = adv_no_hint_df[adv_no_hint_df['level'] == lvl]
        if subset.empty:
            empty_rate_by_level.append(0.0)
            fallback_count_by_level.append(0)
            continue
        empty_rate_by_level.append(float(subset['rag_was_empty_before_guard'].astype(bool).mean()))
        fallback_count_by_level.append(int((subset['rag_answer_source'].astype(str).str.lower() == 'fallback').sum()))

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))

    xg = list(range(len(global_labels)))
    width = 0.36
    axes[0, 0].bar([i - width / 2 for i in xg], naive_global, width, label='Naive', color='#1b9e77')
    axes[0, 0].bar([i + width / 2 for i in xg], adv_global, width, label='Advanced', color='#d95f02')
    axes[0, 0].set_xticks(xg, global_labels)
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].set_title('Accuracy globale Naive vs Advanced')
    axes[0, 0].legend()

    xl = list(range(len(all_levels)))
    if all_levels:
        axes[0, 1].bar([i - width / 2 for i in xl], naive_level_no_hint, width, label='Naive No-Hint', color='#66a61e')
        axes[0, 1].bar([i + width / 2 for i in xl], adv_level_no_hint, width, label='Advanced No-Hint', color='#e6ab02')
        axes[0, 1].set_xticks(xl, all_levels)
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].set_title('No-Hint accuracy per livello')
    axes[0, 1].legend()

    bottoms = [0] * len(all_levels)
    for cat, color in {
        'retrieval_miss': '#7570b3',
        'context_noise': '#e7298a',
        'abstention': '#1f78b4',
        'contradiction': '#e31a1c',
    }.items():
        values = [failure_by_level[lvl][cat] for lvl in all_levels]
        axes[0, 2].bar(all_levels, values, bottom=bottoms, label=cat, color=color)
        bottoms = [b + v for b, v in zip(bottoms, values)]
    axes[0, 2].set_title('Failure decomposition (advanced no-hint)')
    axes[0, 2].legend(fontsize=8)

    axes[1, 0].plot(hybrid_df['level'], hybrid_df['dense'], marker='o', label='dense avg')
    axes[1, 0].plot(hybrid_df['level'], hybrid_df['sparse'], marker='o', label='sparse avg')
    axes[1, 0].plot(hybrid_df['level'], hybrid_df['overlap'], marker='o', label='overlap avg')
    axes[1, 0].set_title('Hybrid retrieval diagnostics per livello')
    axes[1, 0].legend()

    axes[1, 1].bar(all_levels, empty_rate_by_level, color='#a6761d')
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].set_title('Empty-answer rate (pre-guard)')

    axes[1, 2].bar(all_levels, fallback_count_by_level, color='#666666')
    axes[1, 2].set_title('Fallback count per livello')

    plt.tight_layout()
    plt.show()


NameError: name 'naive_benchmark_payload' is not defined

## 12) Ablation runner (stratified 40)

Scaffold per misurazione causale dei componenti advanced sul subset bilanciato 10x livello.

In [12]:
def build_stratified_positions(no_hint_rows: list[dict[str, str]], *, per_level: int = 10) -> list[int]:
    by_level: dict[str, list[int]] = {}
    for pos, row in enumerate(no_hint_rows):
        level = str(row.get('level') or row.get('Livello') or '').strip()
        by_level.setdefault(level, []).append(pos)
    out: list[int] = []
    for level in sorted(by_level.keys(), key=level_sort_key):
        out.extend(by_level[level][:per_level])
    return sorted(set(out))


def _advanced_with_overrides(base_cfg: AdvancedRagConfig, overrides: dict[str, dict[str, Any]]) -> AdvancedRagConfig:
    payload = asdict(base_cfg)
    for section, updates in overrides.items():
        block = payload.get(section)
        if isinstance(block, dict):
            block.update(updates)
        else:
            payload[section] = updates
    return AdvancedRagConfig.from_dict(payload)


ABLATION_CONFIGS: dict[str, dict[str, dict[str, Any]]] = {
    'A0_tuned_full': {},
    'A1_no_rewrite': {'rewrite': {'enabled': False}},
    'A2_metadata_off': {'metadata_filtering': {'mode': 'off', 'enable_heuristics': False}},
    'A3_graph_off': {'graph_expansion': {'enabled': False}},
    'A4_rerank_off': {'rerank': {'enabled': False}},
    'A5_dense_only': {'hybrid': {'enabled': False}},
}


def run_ablation_grid(
    *,
    base_runtime,
    base_config: RagRuntimeConfig,
    configs: dict[str, dict[str, dict[str, Any]]],
    positions: list[int],
) -> dict[str, dict[str, Any]]:
    out: dict[str, dict[str, Any]] = {}
    for label, section_overrides in configs.items():
        tuned_advanced = _advanced_with_overrides(base_config.advanced, section_overrides)
        cfg_i = base_config.with_overrides(advanced=tuned_advanced)
        rt_i = prepare_runtime(cfg_i, client=base_runtime.client)
        try:
            payload_i = run_advanced_benchmark(
                runtime=rt_i,
                mcq_rows=mcq_rows,
                no_hint_rows=no_hint_rows,
                positions=positions,
                api_url=API_URL,
                headers=HEADERS,
                chat_model=CHAT_MODEL,
                judge_model=JUDGE_MODEL,
                timeout_sec=TIMEOUT_SEC,
                post_chat_max_retries=BENCHMARK_POST_CHAT_MAX_RETRIES,
            )
            out[label] = payload_i
        finally:
            rt_i.close()
    return out


# Esecuzione (decommentare quando vuoi lanciare davvero l'ablation):
# ABLATION_POSITIONS = build_stratified_positions(no_hint_rows, per_level=10)
# ablation_payloads = run_ablation_grid(
#     base_runtime=runtime,
#     base_config=config,
#     configs=ABLATION_CONFIGS,
#     positions=ABLATION_POSITIONS,
# )
# ablation_rows = []
# for label, payload in ablation_payloads.items():
#     no_hint_acc = (payload.get('no_hint_summary') or {}).get('accuracy')
#     mcq_acc = (payload.get('mcq_summary') or {}).get('accuracy')
#     diag = payload.get('diagnostics') or {}
#     ablation_rows.append({
#         'label': label,
#         'mcq_accuracy': mcq_acc,
#         'no_hint_accuracy': no_hint_acc,
#         'abstention': (diag.get('no_hint_failure_breakdown') or {}).get('abstention', 0),
#         'retrieval_miss': (diag.get('no_hint_failure_breakdown') or {}).get('retrieval_miss', 0),
#         'reference_hit_rate': diag.get('no_hint_reference_hit_rate'),
#         'graph_enabled_count': diag.get('no_hint_graph_enabled_count'),
#         'graph_positive_count': diag.get('no_hint_graph_retrieved_positive_count'),
#         'avg_rewrite_count': diag.get('no_hint_avg_rewrite_count'),
#     })
# display(pd.DataFrame(ablation_rows).sort_values('label'))


## 13) Cleanup risorse


In [13]:
runtime.close()
if 'naive_runtime' in globals() and naive_runtime is not None:
    naive_runtime.close()
print('Runtime chiuso.')


Runtime chiuso.
